# Modelim i Thjeshtë - Parashikimi i Çmimit

Ky notebook shton një pjesë bonus me modelim, por kodi është mbajtur sa më i thjeshtë. Qëllimi nuk është të krijohet model perfekt, por të tregohet si mund të përdoren të dhënat për të parashikuar çmimin.


In [11]:
from pathlib import Path

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

DATA_PATH = Path("../data/cleaned_airbnb.csv")
OUTPUT_DIR = Path("../outputs/model")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
df.head()


,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,reviews_per_month,calculated_host_listings_count,availability_365,days_since_last_review
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,0.21,6,365,438.0
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,0.38,2,355,224.0
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,0.72,1,365,3200.0
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,4.64,1,194,179.0
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,0.10,1,0,407.0


## 1. Zgjedhja e kolonave

Për modelin përdorim disa kolona që kanë kuptim për çmimin:

- zona
- tipi i dhomës
- minimumi i netëve
- numri i reviews
- reviews për muaj
- numri i listimeve të hostit
- disponueshmëria gjatë vitit


In [12]:
features = [
    "neighbourhood_group",
    "room_type",
    "minimum_nights",
    "number_of_reviews",
    "reviews_per_month",
    "calculated_host_listings_count",
    "availability_365",
]

model_df = df[features + ["price"]].dropna().copy()

# Heqim çmimet 0 dhe çmimet shumë ekstreme që modeli të mos ndikohet shumë.
model_df = model_df[model_df["price"] > 0]
model_df = model_df[model_df["price"] <= model_df["price"].quantile(0.99)]

model_df.head()


,neighbourhood_group,room_type,minimum_nights,number_of_reviews,reviews_per_month,calculated_host_listings_count,availability_365,price
0,Brooklyn,Private room,1,9,0.21,6,365,149
1,Manhattan,Entire home/apt,1,45,0.38,2,355,225
2,Manhattan,Private room,3,0,0.72,1,365,150
3,Brooklyn,Entire home/apt,1,270,4.64,1,194,89
4,Manhattan,Entire home/apt,10,9,0.10,1,0,80


## 2. Kthimi i kolonave kategorike në numra

Modelet e Machine Learning punojnë me numra. Për këtë arsye përdorim `pd.get_dummies()` për kolonat tekstuale si zona dhe tipi i dhomës.


In [13]:
X = model_df[features]
y = model_df["price"]

X = pd.get_dummies(X, columns=["neighbourhood_group", "room_type"], drop_first=True)

X.head()


,minimum_nights,number_of_reviews,reviews_per_month,calculated_host_listings_count,availability_365,neighbourhood_group_Brooklyn,neighbourhood_group_Manhattan,neighbourhood_group_Queens,neighbourhood_group_Staten Island,room_type_Private room,room_type_Shared room
0,1,9,0.21,6,365,True,False,False,False,True,False
1,1,45,0.38,2,355,False,True,False,False,False,False
2,3,0,0.72,1,365,False,True,False,False,True,False
3,1,270,4.64,1,194,True,False,False,False,False,False
4,10,9,0.10,1,0,False,True,False,False,False,False


## 3. Ndarja në train dhe test

80% e të dhënave përdoren për trajnim, ndërsa 20% për testim.


In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)


Train: (36393, 11)
Test: (9099, 11)


## 4. Trajnimi i modelit

Përdorim vetëm një model: `RandomForestRegressor`. Ky model është praktik sepse punon mirë me lidhje jo-lineare dhe nuk kërkon shumë përgatitje shtesë.


In [15]:
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)


,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


## 5. Vlerësimi i modelit

Përdorim tri metrika:

- **MAE**: gabimi mesatar absolut
- **RMSE**: gabimi mesatar me peshë më të madhe për gabime të mëdha
- **R2**: sa mirë modeli shpjegon ndryshimin në çmim


In [16]:
predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

metrics = pd.DataFrame({
    "Metric": ["MAE", "RMSE", "R2"],
    "Value": [mae, rmse, r2]
})

metrics.to_csv(OUTPUT_DIR / "model_metrics.csv", index=False)
metrics


,Metric,Value
0,MAE,35.906163
1,RMSE,49.242334
2,R2,0.456847


## 6. Variablat më të rëndësishme

Kjo tregon cilat kolona kanë ndihmuar më shumë modelin gjatë parashikimit.


In [17]:
importance = pd.DataFrame({
    "feature": X.columns,
    "importance": model.feature_importances_
})

importance = (
    importance
    .sort_values("importance", ascending=False)
    .head(10)
)

importance.to_csv(OUTPUT_DIR / "feature_importance.csv", index=False)
importance


,feature,importance
9,room_type_Private room,0.367171
2,reviews_per_month,0.139801
4,availability_365,0.134548
1,number_of_reviews,0.101107
0,minimum_nights,0.071202
6,neighbourhood_group_Manhattan,0.063347
10,room_type_Shared room,0.063118
3,calculated_host_listings_count,0.050367
5,neighbourhood_group_Brooklyn,0.005944
7,neighbourhood_group_Queens,0.002478


## Interpretim

- Ky model është version i thjeshtë për qëllim demonstrimi.
- Nëse MAE dhe RMSE janë të larta, kjo do të thotë se çmimi është i vështirë të parashikohet vetëm me këto kolona.
- Feature importance ndihmon të kuptohet cilat faktorë ndikojnë më shumë në parashikimin e çmimit.
